In [17]:
from pathlib import Path
from dotenv import load_dotenv

# .env 파일 경로를 안전하게 다루기 위해 Path를 사용한다.
# override=True는 기존 환경변수보다 이 파일의 값을 우선 적용한다.
load_dotenv(dotenv_path=Path("/Users/yoonchan/Desktop/gb/ai/llm/.env"), override=True)

True

In [18]:
# DB 조회와 DataFrame 처리를 위한 라이브러리 불러오기
import os
import pandas as pd
from sqlalchemy import create_engine, text

# .env에 저장한 DB 접속 정보를 SQLAlchemy 접속 문자열로 조합하기
# strip()은 환경변수 끝에 들어갈 수 있는 공백/개행을 제거한다.
db_url = (
    f"postgresql://{os.getenv('DB_USER').strip()}:{os.getenv('DB_PASSWORD').strip()}"
    f"@{os.getenv('DB_HOST').strip()}:{os.getenv('DB_PORT').strip()}/{os.getenv('DB_NAME').strip()}"
)

# pandas.read_sql이 사용할 DB 연결 엔진 만들기
engine = create_engine(db_url)

# 노트북 검증용 로그인 회원 ID
# FastAPI에서는 Spring 인증 객체에서 받은 member_id를 반드시 주입해야 한다.
ACTIVE_MEMBER_ID = 1
ACTIVE_VIDEO_SESSION_ID = None


# RAG 문서로 만들 화상채팅 요약 데이터를 DB에서 조회하기
# summary는 검색 대상 본문이고, video_session_id/caller_id/receiver_id는 범위 제한과 출처 추적에 필요하다.
# 운영 흐름과 동일하게 현재 member_id가 caller 또는 receiver인 화상채팅 요약만 가져온다.
def load_video_summaries_for_member(member_id, video_session_id=None):
    if member_id is None:
        raise ValueError("member_id is required")

    query = text(
        """
        select
            s.id as summary_id,
            s.video_session_id,
            vs.conversation_id,
            vs.caller_id,
            vs.receiver_id,
            s.created_datetime,
            s.summary
        from tbl_ai_video_summary s
        join tbl_video_session vs on vs.id = s.video_session_id
        where (vs.caller_id = :member_id or vs.receiver_id = :member_id)
          and (:video_session_id is null or s.video_session_id = cast(:video_session_id as bigint))
        order by s.video_session_id
        """
    )
    params = {
        "member_id": int(member_id),
        "video_session_id": None if video_session_id is None else int(video_session_id),
    }
    return pd.read_sql(query, engine, params=params)


summary_df = load_video_summaries_for_member(ACTIVE_MEMBER_ID, ACTIVE_VIDEO_SESSION_ID)

# 노트북에서 조회 결과를 바로 확인하기
summary_df


,summary_id,video_session_id,conversation_id,caller_id,receiver_id,created_datetime,summary
0,37,1,1,1,4,2026-05-10 10:59:32.531979,2026-05-01 화상회의 요약 — 발주사 A와의 단가 협상 정례 회의.\n참석자...
1,39,3,12,1,4,2026-05-10 10:59:32.531979,2026-05-03 화상회의 요약 — RAG 챗봇 도입 기술 검토 정기 회의.\n참...


In [19]:
from langchain_community.document_loaders import TextLoader

# 문서 만들기
summaries_dir = Path("./summaries")
summaries_dir.mkdir(exist_ok=True)

expected_files = []
metadata_by_file = {}

# summary 컬럼만 txt 파일에 요약이 들어가므로, 나머지 필요한 컬럼을 metadata에 집어 넣기
for row in summary_df.itertuples(index=False):
    video_session_id = int(row.video_session_id)
    txt_path = summaries_dir / f"session_{video_session_id}.txt"
    txt_path.write_text(row.summary, encoding="utf-8")
    expected_files.append(txt_path)
    metadata_by_file[str(txt_path)] = {
        "summary_id": int(row.summary_id),
        "video_session_id": video_session_id,
        "conversation_id": int(row.conversation_id),
        "caller_id": int(row.caller_id),
        "receiver_id": int(row.receiver_id),
        "created_datetime": str(row.created_datetime),
    }

docs = []
# 만든 문서 불러오기
for txt_file in sorted(expected_files, key=lambda p: int(p.stem.split("_")[1])):
    loader = TextLoader(str(txt_file), encoding="utf-8")
    loaded_docs = loader.load()
    # 로드된  document 객체에 metadata 붙이기
    for doc in loaded_docs:
        doc.metadata.update(metadata_by_file[str(txt_file)])
        doc.metadata["source"] = str(txt_file)
    docs.extend(loaded_docs)

# 폴더에는 있는데, 이번 DB 조회 결과에는 없는 파일 남기기
stale_files = sorted(set(summaries_dir.glob("session_*.txt")) - set(expected_files))
print(f"DB summary 수: {len(summary_df)}")
print(f"문서 수: {len(docs)}")
print(f"로드 제외된 기존 txt 수: {len(stale_files)}")
if docs:
    print(f"첫 문서 metadata: {docs[0].metadata}")
    print(f"첫 문서: {docs[0].page_content}")


DB summary 수: 2
문서 수: 2
로드 제외된 기존 txt 수: 31
첫 문서 metadata: {'source': 'summaries/session_1.txt', 'summary_id': 37, 'video_session_id': 1, 'conversation_id': 1, 'caller_id': 1, 'receiver_id': 4, 'created_datetime': '2026-05-10 10:59:32.531979'}
첫 문서: 2026-05-01 화상회의 요약 — 발주사 A와의 단가 협상 정례 회의.
참석자: 영업본부 김과장(발주사 A 담당), 구매팀 이대리, CFO 박상무, 발주사 A 측 구매책임 J 매니저.
주요 안건은 (1) 기존 공급단가 대비 7% 인상안 정식 제시, (2) 환율 및 원재료(HDPE/철강) 상승분 반영 근거 자료 공유, (3) 최소발주수량(MOQ) 조정.
결정 사항: 기존 단가 대비 정확히 7.0% 인상안을 1차로 발주사에 통보하기로 합의. 발주사는 5.5% 카운터를 제안했으나 우리 측은 자재가 인상 누적치(연 9.4%)를 근거로 7% 고수 방침. CFO는 “6.5% 마지노선”을 내부적으로 합의.
액션 아이템: 김과장은 5월 9일까지 인상 근거 IR 자료(PDF) 송부, 이대리는 MOQ 조정안 시뮬레이션 작성, 박상무는 5월 12일 임원 회의에서 최종 의사결정 후 5월 14일 공식 통지서 발송. 다음 주 임원 회의 후 최종 결정 예정.


## 2. 시맨틱 캐시 설정 (질문 단위)

**출처**: `a_LLM.ipynb` cell `54e5489b`의 Redis 벡터 검색 흐름 기반.

중요: RAG 체인의 LLM 최종 프롬프트에 LangChain 전역 `RedisSemanticCache`를 걸면 긴 공통 프롬프트/Context 때문에 서로 다른 질문도 같은 답변으로 캐시 히트될 수 있다.  
따라서 전역 LLM 캐시는 끄고, `member_id + video_session_id + question`을 임베딩한 답변 캐시를 RAG 앞단에 둔다.


In [20]:
# LangChain 전역 LLM 캐시 제어, 한국어 임베딩, Redis VectorStore를 사용하기 위한 import
from langchain.globals import set_llm_cache
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Redis
from langchain_community.vectorstores.redis import RedisTag
from redis import Redis as RedisClient

# 답변 캐시용 질문 임베딩 모델
# 질문끼리 의미가 비슷한지 비교하는 용도이므로 RAG 검색용 임베딩과 같은 한국어 SBERT 계열을 사용한다.
cache_embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Redis semantic cache 저장 위치와 검색 기준 설정
REDIS_URL = "redis://localhost:6380"
ANSWER_CACHE_INDEX = "video_chat_rag_answer_cache"
ANSWER_CACHE_SCORE_THRESHOLD = 0.1

# Redis 인덱스 스키마
# question은 metadata의 text field, answer/scope는 tag field로 둔다.
# 실제 의미 검색 대상은 texts=[cache_query]로 저장되는 content/content_vector이고, answer는 반환용 metadata다.
# LangChain Redis는 문자열 metadata를 text, 문자열 리스트 metadata를 tag로 자동 추론하므로 저장 타입도 schema와 맞춘다.
ANSWER_CACHE_SCHEMA = {
    "text": [{"name": "question"}],
    "numeric": [],
    "tag": [{"name": "answer"}, {"name": "scope"}],
}

# RAG 최종 프롬프트에는 LangChain 전역 LLM cache를 적용하지 않는다.
# 이 노트북은 사용자/회의 범위를 가진 answer cache를 직접 관리한다.
set_llm_cache(None)


# 운영에서는 member_id가 Spring 인증 객체에서 반드시 들어와야 한다.
# member_id가 없으면 member_all 같은 전역 캐시 scope를 만들지 않고 즉시 실패시킨다.
def require_member_id(member_id):
    if member_id is None:
        raise ValueError("member_id is required")
    return int(member_id)


# 캐시를 사용자와 회의 범위별로 분리하기 위한 scope 문자열 만들기
# video_session_id가 없으면 해당 사용자의 전체 화상채팅 범위, 있으면 특정 회의 범위로 본다.
def build_cache_scope(video_session_id=None, member_id=None):
    member_scope = str(require_member_id(member_id))
    session_scope = "all" if video_session_id is None else str(int(video_session_id))
    return f"member_{member_scope}__video_session_{session_scope}"


# 캐시 검색용 텍스트 만들기
# 같은 질문이라도 scope가 다르면 다른 문서처럼 임베딩되도록 scope를 질문 앞에 붙인다.
def build_cache_query(question, video_session_id=None, member_id=None):
    scope = build_cache_scope(video_session_id=video_session_id, member_id=member_id)
    return f"{scope}\n질문: {question}"


# Redis tag metadata가 문자열 또는 문자열 리스트로 돌아오는 경우를 모두 같은 값으로 읽기
def read_single_tag_value(value):
    if isinstance(value, (list, tuple)):
        return value[0] if value else None
    return value


# 이미 만들어진 Redis 인덱스를 VectorStore 객체로 다시 연결하기
# 조회 시점마다 같은 인덱스 이름과 스키마를 사용해야 이전 캐시를 검색할 수 있다.
def get_answer_cache_store():
    return Redis.from_existing_index(
        cache_embeddings,
        index_name=ANSWER_CACHE_INDEX,
        redis_url=REDIS_URL,
        schema=ANSWER_CACHE_SCHEMA,
    )


# 질문이 기존 semantic cache에 있는지 확인하기
def lookup_answer_cache(question, video_session_id=None, member_id=None):
    cache_query = build_cache_query(question, video_session_id=video_session_id, member_id=member_id)
    expected_scope = build_cache_scope(video_session_id=video_session_id, member_id=member_id)
    try:
        vector_db = get_answer_cache_store()
        # 다른 사용자/다른 회의 캐시가 섞이지 않도록 scope tag로 먼저 제한한다.
        scope_filter = RedisTag("scope") == expected_scope
        results = vector_db.similarity_search_with_score(cache_query, k=1, filter=scope_filter)
    except Exception:
        # 인덱스가 아직 없거나 Redis 연결이 실패하면 캐시 미스로 처리하고 RAG 호출로 넘어간다.
        return None

    if not results:
        return None

    # 가장 가까운 질문 1개가 기준 점수 이하이고 scope도 일치하면 캐시 히트로 인정한다.
    doc, score = results[0]
    cached_scope = read_single_tag_value(doc.metadata.get("scope"))
    if score <= ANSWER_CACHE_SCORE_THRESHOLD and cached_scope == expected_scope:
        return {
            "answer": read_single_tag_value(doc.metadata["answer"]),
            "question": doc.metadata.get("question"),
            "score": score,
            "scope": expected_scope,
        }
    return None


# 캐시 미스 후 LLM/RAG로 생성한 답변을 Redis semantic cache에 저장하기
def store_answer_cache(question, answer, video_session_id=None, member_id=None):
    cache_query = build_cache_query(question, video_session_id=video_session_id, member_id=member_id)
    scope = build_cache_scope(video_session_id=video_session_id, member_id=member_id)
    Redis.from_texts(
        texts=[cache_query],
        embedding=cache_embeddings,
        metadatas=[{"question": question, "answer": [answer], "scope": [scope]}],
        index_name=ANSWER_CACHE_INDEX,
        redis_url=REDIS_URL,
        index_schema=ANSWER_CACHE_SCHEMA,
    )


# 테스트를 위해 Redis 캐시 인덱스를 통째로 삭제하기
# 운영에서는 전체 삭제보다 member/session scope 단위 무효화 전략이 필요하다.
def clear_answer_cache():
    redis_client = RedisClient.from_url(REDIS_URL)
    try:
        redis_client.execute_command("FT.DROPINDEX", ANSWER_CACHE_INDEX, "DD")
        print(f"{ANSWER_CACHE_INDEX} 삭제 완료")
    except Exception as e:
        print(f"삭제할 캐시 인덱스 없음: {e}")


## 3. 문서 분할

**출처**: `e_RAG.ipynb` cell `a906c697` 그대로.

회의 1건 summary가 500자보다 짧으면 회의 1건 = 청크 1개가 정상.

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 긴 회의 요약을 검색하기 좋은 단위로 나누기
# chunk_overlap은 문장이 경계에서 잘려도 앞뒤 문맥이 일부 유지되도록 한다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)

# 실제 FAISS에 들어갈 청크 개수 확인하기
print(f"분할된 청크의수: {len(split_documents)}")

NameError: name 'docs' is not defined

## 4. 임베딩 (RAG 검색용)

**출처**: `e_RAG.ipynb` cell `a906c697` 그대로. 한국어 검증 모델 `jhgan/ko-sbert-nli`, 로컬 CPU.

In [22]:
# RAG 검색용 문서/질문 임베딩 모델
# normalize_embeddings=True는 벡터 크기를 정규화해서 유사도 비교를 안정적으로 만든다.
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

## 5. 벡터 DB (FAISS)

**출처**: `e_RAG.ipynb` cell `a906c697` 그대로.

In [23]:
from langchain_community.vectorstores import FAISS

# 분할된 문서를 임베딩해서 로컬 FAISS 벡터 DB에 적재하기
# 이 벡터 DB는 RAG의 참고 문맥을 찾는 검색 저장소다.
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

## 6. 검색기

**출처**: `e_RAG.ipynb` cell `a906c697` 흐름 유지.

프로젝트 적용: `vectorstore.as_retriever()` 기본 검색기는 유지하되, 실제 질문 함수에서는 `video_session_id`가 들어오면 metadata filter로 해당 화상회의 요약만 검색한다.


In [24]:
# 기본 retriever 만들기
# k=4는 질문당 참고 문서 청크를 최대 4개 가져오겠다는 의미다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


# 질문과 선택적 video_session_id를 받아 관련 문서를 검색하기
def retrieve_documents(question, video_session_id=None, k=4):
    search_kwargs = {"k": k}
    if video_session_id is not None:
        # 특정 회의만 물어볼 때는 해당 video_session_id metadata를 가진 청크만 검색한다.
        search_kwargs["filter"] = {"video_session_id": int(video_session_id)}
    return vectorstore.similarity_search(question, **search_kwargs)


# 검색된 Document 목록을 프롬프트에 넣을 문자열 Context로 합치기
def format_documents(documents):
    return "\n\n".join(doc.page_content for doc in documents)

## 7. 프롬프트

**출처**: `e_RAG.ipynb` cell `a906c697` 그대로. Context 안의 정보만 사용, 자료에 없으면 "확인이 불가능하다".

In [25]:
from langchain_core.prompts import PromptTemplate

# RAG 답변용 프롬프트 템플릿
# 핵심은 Context 안의 정보만 사용하게 제한하고, 모르면 모른다고 답하게 만드는 것이다.
prompt = PromptTemplate.from_template(
    """귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

## 8. LLM

**출처**: `e_RAG.ipynb` cell `a906c697`. 모델은 프로젝트 기준 `gpt-5.4-mini`.

In [26]:
from langchain_openai import ChatOpenAI

# RAG 최종 답변을 생성할 LLM
# cache=False는 LangChain의 전역 LLM 캐시가 아니라 위에서 만든 scope 기반 semantic cache만 쓰겠다는 의미다.
llm = ChatOpenAI(model_name="gpt-5.4-mini", temperature=0, cache=False)

## 9. 체인 (LCEL — 비동기 호환)

**출처**: `e_RAG.ipynb` cell `a906c697` 그대로 + `c_LCEL.ipynb` cell `58af8b3f` 비동기 호출 패턴.

LCEL 체인은 `invoke` / `ainvoke` 둘 다 지원 — 정의는 동일하고, 호출 시점에서만 비동기 여부 결정.  
→ FastAPI 엔드포인트에서 `await chain.ainvoke(...)` 그대로 재사용.

In [27]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


# LCEL 입력에서 question만 꺼내기
# 문자열 하나만 들어오는 경우도 처리해서 테스트와 dict 입력을 모두 허용한다.
def _question_from_input(inputs):
    if isinstance(inputs, dict):
        return inputs["question"]
    return inputs


# LCEL 입력에서 선택적 video_session_id 꺼내기
# 없으면 전체 회의 범위 검색으로 처리한다.
def _video_session_id_from_input(inputs):
    if isinstance(inputs, dict):
        return inputs.get("video_session_id")
    return None


# 질문에 맞는 문서를 검색하고 프롬프트 Context 문자열로 변환하기
def _context_from_input(inputs):
    question = _question_from_input(inputs)
    video_session_id = _video_session_id_from_input(inputs)
    return format_documents(retrieve_documents(question, video_session_id=video_session_id))


# 특정 video_session_id가 현재 member_id의 화상채팅인지 DB에서 확인하기
# cache lookup보다 먼저 실행해서 권한 없는 회의 ID가 캐시/RAG 경로로 들어오지 못하게 한다.
def assert_video_session_access(member_id, video_session_id=None):
    member_id = require_member_id(member_id)
    if video_session_id is None:
        return

    query = text(
        """
        select exists (
            select 1
            from tbl_video_session
            where id = :video_session_id
              and (caller_id = :member_id or receiver_id = :member_id)
        )
        """
    )
    params = {"member_id": member_id, "video_session_id": int(video_session_id)}
    with engine.connect() as conn:
        allowed = conn.execute(query, params).scalar()

    if not allowed:
        raise PermissionError("video_session_id is not accessible for member_id")


# LCEL 체인 구성
# 입력 dict에서 context/question을 만들고, prompt -> llm -> string parser 순서로 답변을 만든다.
chain = (
    {"context": RunnableLambda(_context_from_input), "question": RunnableLambda(_question_from_input)}
    | prompt
    | llm
    | StrOutputParser()
)


# 챗봇 질문 처리 함수
# 1) member/session 권한 검증 2) semantic cache 조회 3) 없으면 RAG 체인 호출 4) 새 답변 캐시 저장 순서로 동작한다.
async def ask_video_chat_rag(question, video_session_id=None, member_id=None):
    member_id = require_member_id(member_id)
    assert_video_session_access(member_id, video_session_id)

    cached = lookup_answer_cache(question, video_session_id=video_session_id, member_id=member_id)
    if cached:
        return {
            "answer": cached["answer"],
            "cache_hit": True,
            "cache_score": cached["score"],
            "cache_scope": cached["scope"],
            "source": "semantic_cache",
        }

    # LCEL 체인에 넘길 입력 payload 만들기
    # video_session_id가 있으면 검색 범위 제한용 metadata filter에 사용된다.
    payload = {"question": question}
    if video_session_id is not None:
        payload["video_session_id"] = int(video_session_id)

    # 캐시 미스면 RAG 체인을 비동기로 호출하고, 답변을 같은 scope에 저장한다.
    answer = await chain.ainvoke(payload)
    store_answer_cache(question, answer, video_session_id=video_session_id, member_id=member_id)
    return {
        "answer": answer,
        "cache_hit": False,
        "cache_score": None,
        "cache_scope": build_cache_scope(video_session_id=video_session_id, member_id=member_id),
        "source": "llm_rag",
    }

print("체인 준비 완료")


체인 준비 완료


## 10. 검증 (비동기 호출)

검증 목표:
1. 같은 질문은 두 번째 호출에서 질문 단위 시맨틱 캐시가 히트한다.
2. 유사 질문은 같은 `video_session_id` 스코프 안에서만 캐시 후보가 된다.
3. 무관 질문은 기존 답변을 재사용하지 않고 RAG+LLM으로 간다.
4. `video_session_id` metadata filter가 다른 회의 요약 혼입을 줄인다.


In [28]:
import time

# 테스트를 깨끗하게 시작하기 위해 기존 answer cache를 삭제한다.
clear_answer_cache()

# 첫 질문은 캐시에 없으므로 RAG + LLM을 호출해야 한다.
question = "단가 협상에서 합의한 단가는 얼마야?"

start = time.time()
result = await ask_video_chat_rag(question, video_session_id=1, member_id=ACTIVE_MEMBER_ID)
print(result["answer"])
print(f"\n소요 시간: {time.time() - start:.4f}초")
print(f"cache_hit: {result['cache_hit']} / source: {result['source']}")
assert result["cache_hit"] is False


video_chat_rag_answer_cache 삭제 완료
합의한 단가는 **기존 단가 대비 정확히 7.0% 인상안**입니다.

소요 시간: 1.3725초
cache_hit: False / source: llm_rag


In [29]:
# 같은 질문을 다시 하면 방금 저장된 semantic cache에 맞아야 한다.
start = time.time()
result = await ask_video_chat_rag(question, video_session_id=1, member_id=ACTIVE_MEMBER_ID)
print(result["answer"])
print(f"\n소요 시간: {time.time() - start:.4f}초")
print(f"cache_hit: {result['cache_hit']} / score: {result['cache_score']}")
assert result["cache_hit"] is True


합의한 단가는 **기존 단가 대비 정확히 7.0% 인상안**입니다.

소요 시간: 0.0299초
cache_hit: True / score: 0.0


In [30]:
# 표현만 바꾼 유사 질문이 semantic cache에 히트하는지 확인한다.
similar = "협상한 단가가 얼마였지?"

start = time.time()
result = await ask_video_chat_rag(similar, video_session_id=1, member_id=ACTIVE_MEMBER_ID)
print(result["answer"])
print(f"\n소요 시간: {time.time() - start:.4f}초")
print(f"cache_hit: {result['cache_hit']} / score: {result['cache_score']}")


합의한 단가는 **기존 단가 대비 정확히 7.0% 인상안**입니다.

소요 시간: 0.0281초
cache_hit: True / score: 0.0344


In [31]:
# 의미가 다른 질문은 기존 단가 협상 캐시에 맞으면 안 된다.
unrelated = "화성 탐사 계획에 대해 알려줘"

start = time.time()
result = await ask_video_chat_rag(unrelated, video_session_id=1, member_id=ACTIVE_MEMBER_ID)
print(result["answer"])
print(f"\n소요 시간: {time.time() - start:.4f}초")
print(f"cache_hit: {result['cache_hit']} / source: {result['source']}")
assert result["cache_hit"] is False, "무관 질문은 캐시 미스여야 한다. 기존 단가 협상 답변 캐시에 히트하면 안 된다."


제공된 정보로는 확인이 불가능하다.  
주어진 문맥은 발주사 A와의 단가 협상, 환율 및 원재료 상승분, MOQ 조정에 관한 회의 요약이며, 화성 탐사 계획에 대한 내용은 포함되어 있지 않습니다.

소요 시간: 1.4431초
cache_hit: False / source: llm_rag


In [32]:
# 현재 ACTIVE_MEMBER_ID가 참여한 회의에 대해 질문을 돌려보며 video_session_id별 검색 범위가 동작하는지 확인한다.
checks = [
    ("RAG 챗봇은 어떤 임베딩 모델을 쓰기로 했어?", 3),
    ("단가 협상에서 인상률은 몇 퍼센트야?", 1),
]

# 각 질문을 실행하고 cache_hit/source/answer를 함께 출력해서 흐름을 비교한다.
for q, video_session_id in checks:
    start = time.time()
    result = await ask_video_chat_rag(q, video_session_id=video_session_id, member_id=ACTIVE_MEMBER_ID)
    print(f"Q: {q}")
    print(f"video_session_id: {video_session_id}")
    print(f"cache_hit: {result['cache_hit']} / source: {result['source']}")
    print(f"A: {result['answer']}")
    print(f"({time.time() - start:.4f}초)\n")

# 권한 없는 회의 ID는 cache lookup이나 RAG 호출 전에 차단되어야 한다.
try:
    await ask_video_chat_rag("UI 개편 회의에서 뭘 정했어?", video_session_id=2, member_id=ACTIVE_MEMBER_ID)
    raise AssertionError("권한 없는 video_session_id가 차단되지 않았다.")
except PermissionError as e:
    print(f"권한 검증 성공: {e}")


Q: RAG 챗봇은 어떤 임베딩 모델을 쓰기로 했어?
video_session_id: 3
cache_hit: False / source: llm_rag
A: RAG 챗봇은 **jhgan/ko-sbert-nli** 임베딩 모델을 쓰기로 했습니다.  
문맥상 **한국어 정확도가 검증된 모델**이며, **768차원**으로 결정되었습니다.
(1.8513초)

Q: 단가 협상에서 인상률은 몇 퍼센트야?
video_session_id: 1
cache_hit: True / source: semantic_cache
A: 합의한 단가는 **기존 단가 대비 정확히 7.0% 인상안**입니다.
(0.0323초)

권한 검증 성공: video_session_id is not accessible for member_id
